# Verify immutable artifact contract pilot


Check all pilot identities and schema compatibility. Synthetic metadata/payload round trips do not measure biological accuracy.

In [1]:
from pathlib import Path
import json, hashlib, ast
from starplast import artifacts as A
from starplast.splits import SplitManifest, Assignment
out = Path('results/artifact_contracts_2026_10_08_v2')
m = json.loads((out / 'manifest.json').read_text())
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
for path, digest in m['input_sha256'].items():
    assert sha(path) == digest, path
for name, digest in m['outputs'].items():
    assert sha(out / name) == digest, name
s = json.loads((out / 'split.json').read_text())
s['assignments'] = tuple(Assignment(**row) for row in s['assignments'])
split = SplitManifest(**s)
verified = []
for mp in out.glob('*/manifest.json'):
    raw = json.loads(mp.read_text())
    spec = A.spec_from_dict(raw['spec'])
    artifact = A.read_artifact(mp.parent, expected=spec, split=split if any(d.kind == 'split' for d in spec.dependencies) else None)
    verified.append(artifact.identity)
assert len(verified) == 10
for path in ('starplast/artifacts.py', 'scripts/freeze_artifact_pilot.py'):
    ast.parse(Path(path).read_text(), feature_version=(3, 10))
{'verified_artifacts': len(verified), 'minimum_python_syntax': '3.10', 'biological_accuracy': 'not measured'}

{'verified_artifacts': 10, 'minimum_python_syntax': '3.10', 'biological_accuracy': 'not measured'}